# Phase 3: Feature Engineering
In this phase, we aim to extract new predictive variables from the raw dataset. [cite_start]According to our project outline, we must rely only on observable physical characteristics to predict a building's energy performance[cite: 26]. 

Our main tasks are:
1. **Deduplication:** Removing multiple EPC records for the same dwelling to prevent data leakage[cite: 35].
2. **Feature Creation:** Calculating `feature_energy_density` and `feature_building_age` proxies.

*Note: Data transformation steps that learn from the data (like scaling) will be strictly handled inside a Cross-Validation Pipeline during Phase 4 to avoid data leakage[cite: 53].*

In [35]:
import pandas as pd
import numpy as np

# Load the raw dataset
# We load the raw dataset directly to preserve columns like BUILDING_REFERENCE_NUMBER and INSPECTION_DATE
df_raw = pd.read_parquet('manchester_epc_prepped.parquet')

print(f"Initial raw dataset size: {len(df_raw)} rows")

Initial raw dataset size: 334270 rows


## 1. Deduplication (Leakage Prevention)
To avoid target leakage caused by updated certificates for the same property, we inspect the dataset for duplicate entries and retain only the most recent record per dwelling[cite: 35].

In [36]:
# Convert inspection date to datetime objects for accurate sorting
df_raw['INSPECTION_DATE'] = pd.to_datetime(df_raw['INSPECTION_DATE'])

# Sort by inspection date and keep the latest certificate for each unique building
df_feat = df_raw.sort_values('INSPECTION_DATE').drop_duplicates(subset=['BUILDING_REFERENCE_NUMBER'], keep='last').copy()

print(f"Dataset size after dropping duplicates: {len(df_feat)} rows")
# We expect this to drop the dataset size to approximately 150,000 records.

Dataset size after dropping duplicates: 266470 rows


# 2. Add flag features


In [37]:
is_old = df_feat['construction_year_estimated'] < 1919

if 'win_glazing_high_perf' in df_feat.columns:
    df_feat['old_with_high_perf_glazing'] = (is_old & (df_feat['win_glazing_high_perf'] == 1)).astype(int)
else:
    df_feat['old_with_high_perf_glazing'] = 0

if 'heat_energy_electric' in df_feat.columns:
    df_feat['old_with_electric_heating'] = (is_old & (df_feat['heat_energy_electric'] == 1)).astype(int)
else:
    df_feat['old_with_electric_heating'] = 0

if 'win_extent_full' in df_feat.columns:
    df_feat['old_with_full_window_replacement'] = (is_old & (df_feat['win_extent_full'] == 1)).astype(int)
else:
    df_feat['old_with_full_window_replacement'] = 0

flag_cols = [c for c in df_feat.columns if c.startswith('old_with_')]

for col in flag_cols:
    counts = df_feat[col].value_counts().sort_index()
    print(f"{col}: 0 → {counts.get(0, 0):,} | 1 → {counts.get(1, 0):,}")

old_with_high_perf_glazing: 0 → 266,169 | 1 → 301
old_with_electric_heating: 0 → 260,784 | 1 → 5,686
old_with_full_window_replacement: 0 → 216,096 | 1 → 50,374


## 3. Final Cleanup & Export (Preventing Data Leakage)
To build a robust model that strictly relies on physical building characteristics, we must remove identification columns and variables that cause target leakage.

Specifically, we drop:
* `ENERGY_CONSUMPTION_CURRENT` — outcome-related, leaky as a feature.
* `CURRENT_ENERGY_RATING` — already converted to `target_is_efficient`.
* ID and Date columns.

**`CURRENT_ENERGY_EFFICIENCY` is kept** as the regression target (`y`), per professor feedback. `target_is_efficient` is kept for post-prediction binning (predicted score ≥ 69 → efficient).

In [38]:
# Drop columns that cause leakage or are just non-predictive IDs
cols_to_drop_final = [
    'BUILDING_REFERENCE_NUMBER', 
    'INSPECTION_DATE', 
    'CURRENT_ENERGY_RATING',       # Replaced by target_is_efficient
    'ENERGY_CONSUMPTION_CURRENT',  # Outcome-related — leaky as a feature
    # CURRENT_ENERGY_EFFICIENCY is kept as the regression target (y)
    # target_is_efficient is kept for post-prediction binning/evaluation
]

df_final = df_feat.drop(columns=[col for col in cols_to_drop_final if col in df_feat.columns])

print("=== PHASE 3 FINAL DATASET ===")
print(f"Final shape ready for Phase 4: {df_final.shape}")
print("\nTarget columns confirmed present:")
print(f"  - CURRENT_ENERGY_EFFICIENCY : min={df_final['CURRENT_ENERGY_EFFICIENCY'].min()}, max={df_final['CURRENT_ENERGY_EFFICIENCY'].max()}")
print(f"  - target_is_efficient counts: {df_final['target_is_efficient'].value_counts().to_dict()}")

output_filename = 'manchester_epc_phase3_final.parquet'
df_final.to_parquet(output_filename)
print(f"\nSUCCESS: Phase 3 is complete! Clean and engineered data saved as '{output_filename}'.")

=== PHASE 3 FINAL DATASET ===
Final shape ready for Phase 4: (266470, 72)

Target columns confirmed present:
  - CURRENT_ENERGY_EFFICIENCY : min=1, max=147
  - target_is_efficient counts: {1: 155584, 0: 110886}

SUCCESS: Phase 3 is complete! Clean and engineered data saved as 'manchester_epc_phase3_final.parquet'.
